In [1]:
# OpenCode / OpenAI-compatible notebook setup.
# Install once in the notebook's Python environment: pip install openai python-dotenv pyyaml
#
###################################################################################################
###################################################################################################
################################################################################################### Standard Modules
#
import os as _nb_os

from pathlib import Path as _NotebookPath

from dotenv import load_dotenv as _notebook_load_dotenv
#
###################################################################################################
###################################################################################################
################################################################################################### Notebook Paths
#
CHAPTER      = "s01_agent_loop"
#
PROJECT_ROOT = next(
    (
        directory
        for start in [_NotebookPath.cwd(), *_NotebookPath.cwd().parents]
        for directory in [start]
        if ((directory / CHAPTER / "code.py").is_file())
    ),
    None,
)
#
if (PROJECT_ROOT is None):

    raise (FileNotFoundError(
        f"Start Jupyter from the project root or {CHAPTER}/ "
        "so code.py can be located"
    ))
#

_notebook_load_dotenv(PROJECT_ROOT / ".env", override=True)
_nb_os.chdir(PROJECT_ROOT)

__file__ = str(PROJECT_ROOT / CHAPTER / "code.py")
#
###################################################################################################
###################################################################################################
################################################################################################### OpenCode Configuration
#
# The existing ANTHROPIC_* environment names remain supported.
_notebook_key   = (_nb_os.getenv("OPENCODE_API_KEY")  or  _nb_os.getenv("ANTHROPIC_API_KEY"))
_notebook_model = (_nb_os.getenv("OPENCODE_MODEL")  or  _nb_os.getenv("MODEL_ID"))
_notebook_url   = (
    _nb_os.getenv("OPENCODE_BASE_URL")
    or  _nb_os.getenv("ANTHROPIC_BASE_URL")
    or  "https://opencode.ai/inference/openai/v1"
).rstrip("/")
#

if (_notebook_url.endswith("/chat/completions")):

    _notebook_url = _notebook_url[:-len("/chat/completions")]
#

if (not _notebook_key):

    raise (RuntimeError(
        "Set OPENCODE_API_KEY (or ANTHROPIC_API_KEY) in the root .env"
    ))
#

if (not _notebook_model):

    raise (RuntimeError(
        "Set MODEL_ID (or OPENCODE_MODEL) in the root .env"
    ))
#

_nb_os.environ["OPENCODE_API_KEY"]   = _notebook_key
_nb_os.environ["OPENCODE_BASE_URL"]  = _notebook_url
_nb_os.environ["ANTHROPIC_BASE_URL"] = _notebook_url
_nb_os.environ["MODEL_ID"]           = _notebook_model

print(f"{CHAPTER}: OpenCode via OpenAI-compatible API, model={_notebook_model}")

s01_agent_loop: OpenCode via OpenAI-compatible API, model=glm-5.3-flash


In [2]:
#!/usr/bin/env python3
"""
s01_agent_loop.py - The Agent Loop

The entire secret of an AI coding agent in one pattern:

    while True:
        response = LLM(messages, tools)
        if response contains no tool_calls:
            break
        execute tools
        append results

    +----------+      +-------+      +---------+
    |   User   | ---> |  LLM  | ---> |  Tool   |
    |  prompt  |      |       |      | execute |
    +----------+      +---+---+      +----+----+
                          ^               |
                          |   tool_result |
                          +---------------+
                          (loop continues)

This is the core loop: feed tool results back to the model
until the model decides to stop. Later chapters add policy,
hooks, and lifecycle controls around it.

Usage:
    pip install openai python-dotenv
    ANTHROPIC_API_KEY=... MODEL_ID=... python s01_agent_loop/code.py
"""
#

'\ns01_agent_loop.py - The Agent Loop\n\nThe entire secret of an AI coding agent in one pattern:\n\n    while True:\n        response = LLM(messages, tools)\n        if response contains no tool_calls:\n            break\n        execute tools\n        append results\n\n    +----------+      +-------+      +---------+\n    |   User   | ---> |  LLM  | ---> |  Tool   |\n    |  prompt  |      |       |      | execute |\n    +----------+      +---+---+      +----+----+\n                          ^               |\n                          |   tool_result |\n                          +---------------+\n                          (loop continues)\n\nThis is the core loop: feed tool results back to the model\nuntil the model decides to stop. Later chapters add policy,\nhooks, and lifecycle controls around it.\n\nUsage:\n    pip install openai python-dotenv\n    ANTHROPIC_API_KEY=... MODEL_ID=... python s01_agent_loop/code.py\n'

In [ ]:
import json
import os
import subprocess
#

try:

    import readline
    # #143 UTF-8 backspace fix for macOS libedit
    readline.parse_and_bind('set bind-tty-special-chars off')
    readline.parse_and_bind('set input-meta on')
    readline.parse_and_bind('set output-meta on')
    readline.parse_and_bind('set convert-meta off')
#
except ImportError:

    pass
#

from dotenv import load_dotenv
from openai import OpenAI
import uuid
#

load_dotenv(override=True)
#

API_KEY  = ((os.getenv("OPENCODE_API_KEY")) or (os.environ["ANTHROPIC_API_KEY"]))
BASE_URL = ((os.getenv("OPENCODE_BASE_URL") or os.environ["ANTHROPIC_BASE_URL"]).rstrip("/"))

if (BASE_URL.endswith("/chat/completions")):

    BASE_URL = BASE_URL[:-len("/chat/completions")]
#

In [ ]:
SESSION_ID = f"ses_{uuid.uuid4().hex}"
client = OpenAI(
    api_key     = API_KEY,
    base_url    = BASE_URL,
    default_headers = {
        "x-opencode-session": SESSION_ID,
        "User-Agent": "learn-claude-code/1.0",
    },
)
#

MODEL  = os.environ["MODEL_ID"]
#

ENVIRONMENT_PROMPT = (
    ("Windows: the bash tool runs through cmd.exe; use cmd.exe syntax, not Unix "
    "Bash or PowerShell syntax")
    if (os.name == "nt")
    else ("Unix-like: the bash tool runs the system shell")
)
#
SYSTEM             = (
    f"You are a coding agent at {os.getcwd()}. Environment: {ENVIRONMENT_PROMPT}. "
    "Use bash to solve tasks. Act, don't explain."
)
#

In [5]:
TOOLS = [{
    "type"     : "function",
    "function" : {
        "name"        : "bash",
        "description" : "Run a shell command.",
        "parameters"  : {
            "type"       : "object",
            "properties" : {"command": {"type": "string"}},
            "required"   : ["command"],
        },
    },
}]
#

In [6]:
def run_bash(command:str) -> str:

    dangerous = ["rm -rf /", "sudo", "shutdown", "reboot", "> /dev/"]
    if (any(d in command for d in dangerous)):

        return ("Error: Dangerous command blocked")
    #
    try:

        r   = subprocess.run(
            command,
            shell          = True,
            cwd            = os.getcwd(),
            capture_output = True,
            text           = True,
            errors         = "replace",
            timeout        = 120,
        )
        #
        out = (r.stdout + r.stderr).strip()
        return ((out[:50000]) if (out) else ("(no output)"))
    #
    except subprocess.TimeoutExpired:

        return ("Error: Timeout (120s)")
    #
    except (FileNotFoundError, OSError) as e:

        return (f"Error: {e}")
    #
#

In [7]:
def agent_loop(messages:list) -> None:

    """
        Send the conversation to OpenCode and execute any requested bash tools.

        Append assistant tool requests and their matching local results to the
        shared history, then repeat until the model returns without tool calls.
    """

    while (True):

        response = client.chat.completions.create(
            model    = MODEL,
            messages = [{"role": "system", "content": SYSTEM}] + messages,
            tools    = TOOLS,
        )
        #

        reply      = response.choices[0].message
        tool_calls = reply.tool_calls or []

        # Preserve tool-call IDs and arguments for the next model request.
        assistant_message = {
            "role"    : "assistant",
            "content" : reply.content,
        }

        if (tool_calls):

            assistant_message["tool_calls"] = [
                {
                    "id"       : block.id,
                    "type"     : "function",
                    "function" : {
                        "name"      : block.function.name,
                        "arguments" : block.function.arguments,
                    },
                } for block in tool_calls
            ]
        #

        messages.append(assistant_message)

        # No tool calls means the model has finished this turn.
        if (not tool_calls):

            return
        #

        # Execute each tool and return its output with the matching call ID.
        for block in tool_calls:

            try:

                if (block.function.name != "bash"):

                    raise (ValueError("Unknown tool"))
                #

                arguments = json.loads(block.function.arguments)
                command   = arguments["command"]

                if (not isinstance(command, str)):

                    raise (ValueError("Command must be a string"))
                #

                print(f"\033[33m$ {command}\033[0m")
                output = run_bash(command)
            #
            except (ValueError, KeyError, TypeError) as e:

                output = f"Error: Invalid tool request: {e}"
            #

            print(output[:200])

            messages.append({
                "role"         : "tool",
                "tool_call_id" : block.id,
                "content"      : output,
            })
        #
    #
#

In [8]:
RUN_AGENT_INTERACTIVELY = 1

if (RUN_AGENT_INTERACTIVELY):

    print("s01: Agent Loop")
    print("Enter a question, press Enter to send. Type q to quit.\n")

    history = []
    while (True):

        try:

            # \001/\002 tell Readline the ANSI escapes have zero display width.
            query = input("\001\033[36m\002s01 >> \001\033[0m\002")
        #
        except (EOFError, KeyboardInterrupt):

            break
        #

        if (query.strip().lower() in ("q", "exit", "")):

            break
        #

        history.append({"role": "user", "content": query})
        agent_loop(history)
        # Print the model's final text response.
        response_content = history[-1]["content"]
        if (isinstance(response_content, str)):

            print(response_content)
        #
        print()
    #
#

s01: Agent Loop
Enter a question, press Enter to send. Type q to quit.

$ echo 'print("Hello, World!")' > hello.py && cat hello.py
print("Hello, World!")
$ python3 hello.py
Hello, World!
Created `hello.py` and verified it runs, printing "Hello, World!"

$ ls -la /home/mostafaroohy/Work/Python/PlayGround_LearnHarness/*.py
-rw-rw-r-- 1 mostafaroohy mostafaroohy 23 Oct 28 18:31 /home/mostafaroohy/Work/Python/PlayGround_LearnHarness/hello.py
There is 1 Python file in this directory:

- `hello.py` (23 bytes, created earlier)

